# 🤖 AI Functions Showcase - The Art of the Possible

**COMPREHENSIVE AI DEMONSTRATION** - Run this after notebook 01.

## What this notebook demonstrates:
- ✅ **ai_classify** - Intelligent priority and category classification
- ✅ **ai_extract** - Structured data extraction from unstructured text  
- ✅ **ai_gen** - Complex analysis, summaries, and creative content generation
- ✅ **Final Summary Table** - Complete AI-powered ticket analysis

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Showcase:
- `ai_classify` - For priority and category classification
- `ai_extract` - For structured data extraction  
- `ai_gen` - For complex analysis and summaries

---

## 🎯 Goal: Demonstrate the full power of Databricks AI Functions


In [ ]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🚀 Ready to showcase Databricks AI Functions!")


In [ ]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample ticket data:")
display(df_tickets.select("ticket_id", "short_description", "description").limit(3))


# 🎯 AI Function #1: ai_classify

**Purpose:** Intelligent classification of text into predefined categories

**Use Cases:** Priority classification, category assignment, sentiment analysis, risk assessment

**Example:** Classify ticket priorities based on description content


In [ ]:
# AI Function #1: ai_classify - Priority Classification
print("🎯 Demonstrating ai_classify for priority classification...")

# Register DataFrame as temporary view for SQL access
df_tickets.createOrReplaceTempView("tickets")

# Use ai_classify to classify ticket priorities
df_with_priority = spark.sql("""
    SELECT
        *,
        ai_classify(
            description, 
            ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')
        ) as ai_priority_classification
    FROM tickets
""")

print("✅ ai_classify completed - Priority classification done!")
print("\n📊 Priority Classification Results:")
display(df_with_priority.select("ticket_id", "short_description", "ai_priority_classification").limit(5))

# Show distribution of AI classifications
print("\n📈 Priority Distribution:")
df_with_priority.groupBy("ai_priority_classification").count().orderBy(desc("count")).show()


# 🎯 AI Function #2: ai_extract

**Purpose:** Extract structured data from unstructured text using schema definition

**Use Cases:** Action items extraction, contact information parsing, date/time extraction, entity recognition

**Example:** Extract specific action items and requirements from ticket descriptions


In [ ]:
# AI Function #2: ai_extract - Action Items Extraction
print("🎯 Demonstrating ai_extract for structured data extraction...")

# Register DataFrame as temporary view for SQL access
df_with_priority.createOrReplaceTempView("tickets_with_priority")

# Define the schema for extraction
extraction_schema = {
    "action_items": "array<string>",
    "main_requirement": "string",
    "urgency_level": "string",
    "affected_systems": "array<string>"
}

# Use ai_extract to extract structured data
df_with_extraction = spark.sql(f"""
    SELECT
        *,
        ai_extract(
            description,
            '{json.dumps(extraction_schema)}'
        ) as ai_extracted_data
    FROM tickets_with_priority
""")

print("✅ ai_extract completed - Structured data extraction done!")
print("\n📊 Extraction Results:")
display(df_with_extraction.select("ticket_id", "short_description", "ai_extracted_data").limit(3))

# Parse the extracted data for better display
df_parsed = df_with_extraction.withColumn(
    "action_items", 
    col("ai_extracted_data.action_items")
).withColumn(
    "main_requirement", 
    col("ai_extracted_data.main_requirement")
).withColumn(
    "urgency_level", 
    col("ai_extracted_data.urgency_level")
).withColumn(
    "affected_systems", 
    col("ai_extracted_data.affected_systems")
)

print("\n📋 Parsed Extraction Results:")
display(df_parsed.select("ticket_id", "action_items", "main_requirement", "urgency_level").limit(3))


# 🎯 AI Function #3: ai_gen

**Purpose:** Generate creative content, summaries, and complex analysis

**Use Cases:** Executive summaries, detailed analysis, creative content, recommendations

**Example:** Generate comprehensive ticket analysis and recommendations


In [ ]:
# AI Function #3: ai_gen - Comprehensive Analysis
print("🎯 Demonstrating ai_gen for comprehensive analysis...")

# Register DataFrame as temporary view for SQL access
df_parsed.createOrReplaceTempView("tickets_with_extraction")

# Use ai_gen to create comprehensive analysis
df_with_analysis = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Analyze this IT ticket and provide: ',
                '1. Executive summary (2-3 sentences), ',
                '2. Technical complexity (Low/Medium/High), ',
                '3. Estimated effort (hours), ',
                '4. Risk assessment (Low/Medium/High), ',
                '5. Recommended next steps. ',
                'Ticket: ', short_description, ' - ', description
            )
        ) as ai_comprehensive_analysis
    FROM tickets_with_extraction
""")

print("✅ ai_gen completed - Comprehensive analysis done!")
print("\n📊 Analysis Results:")
display(df_with_analysis.select("ticket_id", "short_description", "ai_comprehensive_analysis").limit(3))


# 🎉 Final Summary Table - The Art of the Possible

**Complete AI-powered ticket analysis showcasing all three AI functions**


In [ ]:
# Create Final Summary Table
print("🎉 Creating comprehensive summary table...")

# Create a clean summary table with all AI results
df_final_summary = df_with_analysis.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "action_items",
    "main_requirement", 
    "urgency_level",
    "affected_systems",
    "ai_comprehensive_analysis"
).withColumn(
    "ai_showcase_timestamp", 
    current_timestamp()
)

print("✅ Final summary table created!")
print(f"📊 Total tickets analyzed: {df_final_summary.count()}")

# Display the comprehensive results
print("\n🎯 COMPLETE AI SHOWCASE RESULTS:")
print("="*80)
display(df_final_summary.limit(5))

# Save to Unity Catalog
print("\n💾 Saving results to Unity Catalog...")
df_final_summary.write.format("delta").mode("overwrite").saveAsTable(TABLES["ai_showcase_results"])
print(f"✅ Results saved to: {TABLES['ai_showcase_results']}")

# Show summary statistics
print("\n📈 AI Showcase Summary Statistics:")
print(f"🎯 Tickets processed: {df_final_summary.count()}")
print(f"🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)")
print(f"📊 Data saved to Unity Catalog: {TABLES['ai_showcase_results']}")

print("\n" + "="*80)
print("🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!")
print("="*80)
print("✅ ai_classify: Priority classification")
print("✅ ai_extract: Structured data extraction") 
print("✅ ai_gen: Comprehensive analysis")
print("✅ Final summary table created and saved")
print("="*80)
